# 01 — Exploratory Data Analysis
**Financial PhraseBank — Sentiment Analysis**

Dataset : `data/all-data.csv` — 4 846 lignes, 3 classes (positive / neutral / negative)

Objectifs :
- Distribution des classes
- Distribution de la longueur des phrases
- WordCloud par sentiment
- Top mots par classe (sans stopwords)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

## 1. Chargement des données

In [ ]:
df = pd.read_csv(
    '../data/all-data.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'headline']
)

print(f'Shape : {df.shape}')
print(f'Valeurs manquantes :\n{df.isnull().sum()}')
df.head(10)

In [ ]:
# Longueur en nombre de mots
df['n_words'] = df['headline'].apply(lambda x: len(str(x).split()))
df['n_chars'] = df['headline'].apply(lambda x: len(str(x)))

df.describe(include='all')

## 2. Distribution des classes

In [ ]:
label_order = ['positive', 'neutral', 'negative']
palette     = {'positive': '#2ecc71', 'neutral': '#3498db', 'negative': '#e74c3c'}

counts = df['sentiment'].value_counts().reindex(label_order)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(counts.index, counts.values,
            color=[palette[l] for l in counts.index], edgecolor='white', width=0.55)
for i, (label, v) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, v + 20, f'{v}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Distribution des classes', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Nombre de phrases')
axes[0].set_ylim(0, counts.max() * 1.15)

# Pie chart
axes[1].pie(
    counts.values,
    labels=counts.index,
    colors=[palette[l] for l in counts.index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[1].set_title('Répartition en %', fontsize=13, fontweight='bold')

plt.suptitle('Financial PhraseBank — Vue globale des classes', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('\nNote : dataset déséquilibré (classe neutral sur-représentée)')

## 3. Longueur des phrases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogramme nombre de mots
for label in label_order:
    subset = df[df['sentiment'] == label]['n_words']
    axes[0].hist(subset, bins=30, alpha=0.55,
                 color=palette[label], label=label, edgecolor='white')
axes[0].set_title('Distribution — Nombre de mots', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# Boxplot par classe
data_bp = [df[df['sentiment'] == l]['n_words'].values for l in label_order]
bp = axes[1].boxplot(data_bp, patch_artist=True, labels=label_order,
                     medianprops=dict(color='black', linewidth=2))
for patch, label in zip(bp['boxes'], label_order):
    patch.set_facecolor(palette[label])
    patch.set_alpha(0.7)
axes[1].set_title('Boxplot — Longueur par sentiment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Nombre de mots')

plt.tight_layout()
plt.show()

# Stats
print(df.groupby('sentiment')['n_words'].describe().round(1))

## 4. WordCloud par sentiment

In [ ]:
def get_text_for_class(label):
    texts = df[df['sentiment'] == label]['headline'].tolist()
    combined = ' '.join(str(t) for t in texts)
    # Suppression ponctuation et stopwords
    words = re.findall(r'\b[a-zA-Z]+\b', combined.lower())
    words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return ' '.join(words)

wc_colors = {
    'positive': 'Greens',
    'neutral':  'Blues',
    'negative': 'Reds'
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, label in zip(axes, label_order):
    text = get_text_for_class(label)
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap=wc_colors[label],
        max_words=80,
        collocations=False
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'Sentiment : {label.upper()}', fontsize=13, fontweight='bold',
                 color=palette[label])
    ax.axis('off')

plt.suptitle('WordCloud par classe de sentiment', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 5. Top 20 mots par classe

In [ ]:
def top_words(label, n=20):
    text = get_text_for_class(label)
    counter = Counter(text.split())
    return counter.most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for ax, label in zip(axes, label_order):
    top = top_words(label, 20)
    words, freqs = zip(*top)
    colors = [palette[label]] * len(words)
    ax.barh(list(words)[::-1], list(freqs)[::-1],
            color=colors, edgecolor='white', alpha=0.85)
    ax.set_title(f'Top 20 mots — {label.upper()}', fontsize=12, fontweight='bold',
                 color=palette[label])
    ax.set_xlabel('Fréquence')

plt.suptitle('Top mots par classe (sans stopwords)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Corrélation longueur × sentiment

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label in label_order:
    subset = df[df['sentiment'] == label]
    sns.kdeplot(subset['n_words'], ax=ax, label=label,
                color=palette[label], fill=True, alpha=0.25)
ax.set_title('Densité de la longueur des phrases par sentiment', fontsize=12, fontweight='bold')
ax.set_xlabel('Nombre de mots')
ax.legend()
plt.tight_layout()
plt.show()

print('\n=== Résumé EDA ===')
print(f'Total phrases     : {len(df)}')
print(f'Classes           : {df["sentiment"].unique()}')
print(f'Longueur moyenne  : {df["n_words"].mean():.1f} mots (std={df["n_words"].std():.1f})')
print(f'Min / Max mots    : {df["n_words"].min()} / {df["n_words"].max()}')
print('\nDistribution :')
print(df['sentiment'].value_counts().to_string())